In [ ]:
from __future__ import annotations

import argparse
import ast
import html
import json
import re
from pathlib import Path

import pandas as pd


def resolve_project_root() -> Path:
    """Locate the repository root from the notebook's current working directory."""

    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "raw_data").exists() or (candidate / ".git").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the CineIQ project root from the current working directory. "
        f"Started from: {cwd}. Open the notebook from inside the cloned repository so "
        "raw_data, cleaned_data, and model_artifacts stay inside the project."
    )


PROJECT_ROOT = resolve_project_root()
DEFAULT_BASE = PROJECT_ROOT / "raw_data"
DEFAULT_ML25M = DEFAULT_BASE / "ml-25m"
DEFAULT_OUTPUT = PROJECT_ROOT / "cleaned_data"


def clean_text(value: object) -> str:
    """Normalize review/tag/title text while preserving useful words."""
    if pd.isna(value):
        return ""
    text = html.unescape(str(value))
    text = re.sub(r"<br\s*/?>", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def parse_json_list(value: object) -> list[dict]:
    """Safely parse TMDB JSON-ish list columns."""
    if pd.isna(value) or value == "":
        return []
    try:
        parsed = ast.literal_eval(str(value))
    except (ValueError, SyntaxError):
        return []
    return parsed if isinstance(parsed, list) else []


def names_from_people(people: list[dict], limit: int | None = None) -> list[str]:
    names = [str(person.get("name", "")).strip() for person in people]
    names = [name for name in names if name]
    return names[:limit] if limit else names


def jobs_from_crew(crew: list[dict], jobs: set[str], limit: int | None = None) -> list[str]:
    names = [
        str(person.get("name", "")).strip()
        for person in crew
        if str(person.get("job", "")).strip() in jobs
    ]
    names = [name for name in names if name]
    return names[:limit] if limit else names


def write_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"Wrote {path} ({len(df):,} rows)")


def clean_movies(movies_path: Path, output_dir: Path) -> pd.DataFrame:
    movies = pd.read_csv(movies_path)
    movies = movies.drop_duplicates(subset=["movieId"]).copy()

    movies["title"] = movies["title"].map(clean_text)
    movies["year"] = movies["title"].str.extract(r"\((\d{4})\)\s*$")[0]
    movies["clean_title"] = movies["title"].str.replace(r"\s*\(\d{4}\)\s*$", "", regex=True)
    movies["genres"] = movies["genres"].fillna("")
    movies["genres_list"] = movies["genres"].replace("(no genres listed)", "").str.replace("|", " ", regex=False)
    movies["primary_genre"] = movies["genres"].apply(
        lambda x: "" if x == "(no genres listed)" else str(x).split("|")[0]
    )
    movies["genre_count"] = movies["genres"].apply(
        lambda x: 0 if x == "(no genres listed)" or pd.isna(x) else len(str(x).split("|"))
    )
    movies["year"] = pd.to_numeric(movies["year"], errors="coerce").astype("Int64")

    keep = ["movieId", "title", "clean_title", "year", "genres", "genres_list", "primary_genre", "genre_count"]
    movies = movies[keep].sort_values("movieId")
    write_csv(movies, output_dir / "movies_clean.csv")
    return movies


def clean_links(links_path: Path, output_dir: Path) -> pd.DataFrame:
    links = pd.read_csv(links_path)
    links = links.drop_duplicates(subset=["movieId"]).copy()
    links["imdbId"] = pd.to_numeric(links["imdbId"], errors="coerce").astype("Int64")
    links["tmdbId"] = pd.to_numeric(links["tmdbId"], errors="coerce").astype("Int64")
    links["imdb_url_id"] = links["imdbId"].apply(lambda x: f"tt{int(x):07d}" if pd.notna(x) else "")
    write_csv(links, output_dir / "links_clean.csv")
    return links


def clean_imdb_reviews(imdb_path: Path, output_dir: Path, sample_rows: int | None) -> pd.DataFrame:
    reviews = pd.read_csv(imdb_path, nrows=sample_rows)
    reviews = reviews.drop_duplicates().dropna(subset=["review", "sentiment"]).copy()
    reviews["review_clean"] = reviews["review"].map(clean_text)
    reviews["sentiment"] = reviews["sentiment"].str.lower().str.strip()
    reviews = reviews[reviews["sentiment"].isin(["positive", "negative"])]
    reviews["sentiment_label"] = reviews["sentiment"].map({"negative": 0, "positive": 1})
    reviews["review_length"] = reviews["review_clean"].str.split().str.len()
    reviews = reviews[reviews["review_length"] > 2]
    reviews = reviews[["review_clean", "sentiment", "sentiment_label", "review_length"]]
    write_csv(reviews, output_dir / "imdb_reviews_clean.csv")
    return reviews


def clean_tmdb_credits(tmdb_path: Path, output_dir: Path, sample_rows: int | None) -> pd.DataFrame:
    credits = pd.read_csv(tmdb_path, nrows=sample_rows)
    credits = credits.drop_duplicates(subset=["movie_id"]).copy()
    credits["title"] = credits["title"].map(clean_text)

    cast_lists = credits["cast"].apply(parse_json_list)
    crew_lists = credits["crew"].apply(parse_json_list)

    credits["top_cast"] = cast_lists.apply(lambda people: "|".join(names_from_people(people, 5)))
    credits["cast_count"] = cast_lists.apply(len)
    credits["director"] = crew_lists.apply(lambda crew: "|".join(jobs_from_crew(crew, {"Director"})))
    credits["writers"] = crew_lists.apply(
        lambda crew: "|".join(jobs_from_crew(crew, {"Writer", "Screenplay", "Story"}, 5))
    )
    credits["producers"] = crew_lists.apply(lambda crew: "|".join(jobs_from_crew(crew, {"Producer"}, 5)))
    credits["composer"] = crew_lists.apply(lambda crew: "|".join(jobs_from_crew(crew, {"Original Music Composer"}, 3)))

    keep = ["movie_id", "title", "top_cast", "cast_count", "director", "writers", "producers", "composer"]
    credits = credits[keep].rename(columns={"movie_id": "tmdbId", "title": "tmdb_title"})
    credits["tmdbId"] = pd.to_numeric(credits["tmdbId"], errors="coerce").astype("Int64")
    write_csv(credits, output_dir / "tmdb_credits_clean.csv")
    return credits


def clean_ratings(
    ratings_path: Path,
    output_dir: Path,
    sample_rows: int | None,
    chunksize: int,
    write_full_ratings: bool,
) -> pd.DataFrame:
    stats_parts = []
    cleaned_path = output_dir / "ratings_clean.csv"
    if write_full_ratings and cleaned_path.exists():
        cleaned_path.unlink()

    rows_seen = 0
    reader = pd.read_csv(ratings_path, chunksize=chunksize)
    for chunk_no, chunk in enumerate(reader, start=1):
        if sample_rows is not None:
            remaining = sample_rows - rows_seen
            if remaining <= 0:
                break
            chunk = chunk.head(remaining)
        rows_seen += len(chunk)

        chunk = chunk.dropna(subset=["userId", "movieId", "rating", "timestamp"]).copy()
        chunk["userId"] = pd.to_numeric(chunk["userId"], errors="coerce").astype("Int64")
        chunk["movieId"] = pd.to_numeric(chunk["movieId"], errors="coerce").astype("Int64")
        chunk["rating"] = pd.to_numeric(chunk["rating"], errors="coerce")
        chunk["timestamp"] = pd.to_numeric(chunk["timestamp"], errors="coerce").astype("Int64")
        chunk = chunk.dropna(subset=["userId", "movieId", "rating", "timestamp"])
        chunk = chunk[(chunk["rating"] >= 0.5) & (chunk["rating"] <= 5.0)]
        chunk = chunk.drop_duplicates(subset=["userId", "movieId"], keep="last")
        chunk["rating_datetime"] = pd.to_datetime(chunk["timestamp"], unit="s")

        if write_full_ratings:
            chunk.to_csv(cleaned_path, index=False, mode="a", header=chunk_no == 1)

        part = chunk.groupby("movieId").agg(
            rating_count=("rating", "size"),
            rating_sum=("rating", "sum"),
            rating_mean=("rating", "mean"),
            first_rating_at=("rating_datetime", "min"),
            last_rating_at=("rating_datetime", "max"),
        )
        stats_parts.append(part.reset_index())
        print(f"Processed ratings chunk {chunk_no}: {rows_seen:,} rows")

    stats = pd.concat(stats_parts, ignore_index=True)
    stats = stats.groupby("movieId").agg(
        rating_count=("rating_count", "sum"),
        rating_sum=("rating_sum", "sum"),
        first_rating_at=("first_rating_at", "min"),
        last_rating_at=("last_rating_at", "max"),
    )
    stats["rating_mean"] = stats["rating_sum"] / stats["rating_count"]
    stats = stats.reset_index()
    stats["rating_mean"] = stats["rating_mean"].round(4)
    stats = stats[["movieId", "rating_count", "rating_mean", "first_rating_at", "last_rating_at"]]
    write_csv(stats, output_dir / "movie_rating_stats.csv")
    return stats


def clean_tags(tags_path: Path, output_dir: Path, sample_rows: int | None, chunksize: int) -> pd.DataFrame:
    tag_parts = []
    clean_path = output_dir / "tags_clean.csv"
    if clean_path.exists():
        clean_path.unlink()

    rows_seen = 0
    reader = pd.read_csv(tags_path, chunksize=chunksize)
    for chunk_no, chunk in enumerate(reader, start=1):
        if sample_rows is not None:
            remaining = sample_rows - rows_seen
            if remaining <= 0:
                break
            chunk = chunk.head(remaining)
        rows_seen += len(chunk)

        chunk = chunk.dropna(subset=["userId", "movieId", "tag", "timestamp"]).copy()
        chunk["tag_clean"] = chunk["tag"].map(clean_text).str.lower()
        chunk = chunk[chunk["tag_clean"] != ""]
        chunk["tag_datetime"] = pd.to_datetime(chunk["timestamp"], unit="s")
        chunk = chunk[["userId", "movieId", "tag_clean", "tag_datetime"]].drop_duplicates()
        chunk.to_csv(clean_path, index=False, mode="a", header=chunk_no == 1)

        counts = chunk.groupby(["movieId", "tag_clean"]).size().reset_index(name="count")
        tag_parts.append(counts)
        print(f"Processed tags chunk {chunk_no}: {rows_seen:,} rows")

    tag_counts = pd.concat(tag_parts, ignore_index=True)
    tag_counts = tag_counts.groupby(["movieId", "tag_clean"], as_index=False)["count"].sum()
    tag_counts = tag_counts.sort_values(["movieId", "count", "tag_clean"], ascending=[True, False, True])

    top_tags = (
        tag_counts.groupby("movieId")
        .head(10)
        .groupby("movieId")["tag_clean"]
        .apply(lambda tags: "|".join(tags))
        .reset_index(name="user_tags_top10")
    )
    write_csv(top_tags, output_dir / "movie_user_tags_top10.csv")
    return top_tags


def clean_genome(
    genome_scores_path: Path,
    genome_tags_path: Path,
    output_dir: Path,
    sample_rows: int | None,
    relevance_threshold: float,
    chunksize: int,
) -> pd.DataFrame:
    genome_tags = pd.read_csv(genome_tags_path)
    genome_tags["tag"] = genome_tags["tag"].map(clean_text).str.lower()

    parts = []
    rows_seen = 0
    reader = pd.read_csv(genome_scores_path, chunksize=chunksize)
    for chunk_no, chunk in enumerate(reader, start=1):
        if sample_rows is not None:
            remaining = sample_rows - rows_seen
            if remaining <= 0:
                break
            chunk = chunk.head(remaining)
        rows_seen += len(chunk)

        chunk = chunk[pd.to_numeric(chunk["relevance"], errors="coerce") >= relevance_threshold].copy()
        if not chunk.empty:
            parts.append(chunk)
        print(f"Processed genome chunk {chunk_no}: {rows_seen:,} rows")

    if parts:
        genome = pd.concat(parts, ignore_index=True)
        genome = genome.merge(genome_tags, on="tagId", how="left")
        genome = genome.sort_values(["movieId", "relevance"], ascending=[True, False])
        genome_top = (
            genome.groupby("movieId")
            .head(20)
            .groupby("movieId")["tag"]
            .apply(lambda tags: "|".join(tags.dropna().astype(str)))
            .reset_index(name="genome_tags_top20")
        )
    else:
        genome_top = pd.DataFrame(columns=["movieId", "genome_tags_top20"])

    write_csv(genome_top, output_dir / "movie_genome_tags_top20.csv")
    return genome_top


def build_master_table(
    movies: pd.DataFrame,
    links: pd.DataFrame,
    ratings: pd.DataFrame,
    user_tags: pd.DataFrame,
    genome_tags: pd.DataFrame,
    credits: pd.DataFrame,
    output_dir: Path,
) -> pd.DataFrame:
    master = movies.merge(links, on="movieId", how="left")
    master = master.merge(ratings, on="movieId", how="left")
    master = master.merge(user_tags, on="movieId", how="left")
    master = master.merge(genome_tags, on="movieId", how="left")
    master = master.merge(credits, on="tmdbId", how="left")

    text_cols = [
        "clean_title",
        "genres_list",
        "user_tags_top10",
        "genome_tags_top20",
        "top_cast",
        "director",
        "writers",
        "composer",
    ]
    for col in text_cols:
        if col in master.columns:
            master[col] = master[col].fillna("")

    master["content_features"] = (
        master["clean_title"].astype(str)
        + " "
        + master["genres_list"].astype(str)
        + " "
        + master["user_tags_top10"].astype(str).str.replace("|", " ", regex=False)
        + " "
        + master["genome_tags_top20"].astype(str).str.replace("|", " ", regex=False)
        + " "
        + master["top_cast"].astype(str).str.replace("|", " ", regex=False)
        + " "
        + master["director"].astype(str).str.replace("|", " ", regex=False)
        + " "
        + master["writers"].astype(str).str.replace("|", " ", regex=False)
    ).map(clean_text)

    master["rating_count"] = master["rating_count"].fillna(0).astype(int)
    master["rating_mean"] = master["rating_mean"].fillna(0).round(4)
    master = master.sort_values(["rating_count", "rating_mean"], ascending=[False, False])
    write_csv(master, output_dir / "movies_master.csv")
    return master


def write_data_dictionary(output_dir: Path, args: argparse.Namespace, outputs: dict[str, pd.DataFrame]) -> None:
    dictionary = {
        "project": "CINEIQ movie recommender dataset cleaning",
        "settings": {
            "sample_rows": args.sample_rows,
            "ratings_chunksize": args.chunksize,
            "write_full_ratings": args.write_full_ratings,
            "genome_relevance_threshold": args.genome_relevance_threshold,
        },
        "outputs": {
            name: {
                "rows": int(len(df)),
                "columns": list(df.columns),
            }
            for name, df in outputs.items()
        },
    }
    path = output_dir / "data_dictionary.json"
    path.write_text(json.dumps(dictionary, indent=2, default=str), encoding="utf-8")
    print(f"Wrote {path}")


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Clean CINEIQ movie recommendation datasets.")
    parser.add_argument("--base-dir", type=Path, default=DEFAULT_BASE)
    parser.add_argument("--ml25m-dir", type=Path, default=DEFAULT_ML25M)
    parser.add_argument("--output-dir", type=Path, default=DEFAULT_OUTPUT)
    parser.add_argument("--sample-rows", type=int, default=None, help="Read only N rows from large files for a quick test.")
    parser.add_argument("--chunksize", type=int, default=1_000_000)
    parser.add_argument("--write-full-ratings", action="store_true", help="Also write cleaned ratings_clean.csv.")
    parser.add_argument("--genome-relevance-threshold", type=float, default=0.70)
    # In Jupyter/IPython, the kernel injects arguments such as
    # "-f /path/to/kernel.json". parse_known_args keeps the script usable
    # both from Terminal and from a notebook cell.
    args, _unknown = parser.parse_known_args()
    return args


def main() -> None:
    args = parse_args()
    output_dir = args.output_dir
    output_dir.mkdir(parents=True, exist_ok=True)

    tmdb_path = args.base_dir / "tmdb_5000_credits.csv"
    imdb_path = args.base_dir / "IMDB Dataset.csv"
    movies_path = args.ml25m_dir / "movies.csv"
    links_path = args.ml25m_dir / "links.csv"
    ratings_path = args.ml25m_dir / "ratings.csv"
    tags_path = args.ml25m_dir / "tags.csv"
    genome_scores_path = args.ml25m_dir / "genome-scores.csv"
    genome_tags_path = args.ml25m_dir / "genome-tags.csv"

    movies = clean_movies(movies_path, output_dir)
    links = clean_links(links_path, output_dir)
    reviews = clean_imdb_reviews(imdb_path, output_dir, args.sample_rows)
    credits = clean_tmdb_credits(tmdb_path, output_dir, args.sample_rows)
    ratings = clean_ratings(ratings_path, output_dir, args.sample_rows, args.chunksize, args.write_full_ratings)
    user_tags = clean_tags(tags_path, output_dir, args.sample_rows, args.chunksize)
    genome_tags = clean_genome(
        genome_scores_path,
        genome_tags_path,
        output_dir,
        args.sample_rows,
        args.genome_relevance_threshold,
        args.chunksize,
    )
    master = build_master_table(movies, links, ratings, user_tags, genome_tags, credits, output_dir)

    write_data_dictionary(
        output_dir,
        args,
        {
            "movies_clean": movies,
            "links_clean": links,
            "imdb_reviews_clean": reviews,
            "tmdb_credits_clean": credits,
            "movie_rating_stats": ratings,
            "movie_user_tags_top10": user_tags,
            "movie_genome_tags_top20": genome_tags,
            "movies_master": master,
        },
    )

    print(f"\nDone. Main file for the recommender: {output_dir / 'movies_master.csv'}")


if __name__ == "__main__":
    main()


Wrote D:\projects\CineIQ\cleaned_data\movies_clean.csv (62,423 rows)
Wrote D:\projects\CineIQ\cleaned_data\links_clean.csv (62,423 rows)
Wrote D:\projects\CineIQ\cleaned_data\imdb_reviews_clean.csv (49,582 rows)
Wrote D:\projects\CineIQ\cleaned_data\tmdb_credits_clean.csv (4,803 rows)
Processed ratings chunk 1: 1,000,000 rows
Processed ratings chunk 2: 2,000,000 rows
Processed ratings chunk 3: 3,000,000 rows
Processed ratings chunk 4: 4,000,000 rows
Processed ratings chunk 5: 5,000,000 rows
Processed ratings chunk 6: 6,000,000 rows
Processed ratings chunk 7: 7,000,000 rows
Processed ratings chunk 8: 8,000,000 rows
Processed ratings chunk 9: 9,000,000 rows
Processed ratings chunk 10: 10,000,000 rows
Processed ratings chunk 11: 11,000,000 rows
Processed ratings chunk 12: 12,000,000 rows
Processed ratings chunk 13: 13,000,000 rows
Processed ratings chunk 14: 14,000,000 rows
Processed ratings chunk 15: 15,000,000 rows
Processed ratings chunk 16: 16,000,000 rows
Processed ratings chunk 17: 

In [ ]:
import pandas as pd

from pathlib import Path

PROJECT_ROOT = resolve_project_root()
movies = pd.read_csv(PROJECT_ROOT / "cleaned_data" / "movies_master.csv", low_memory=False)
movies.head(10)



In [ ]:
from pathlib import Path

PROJECT_ROOT = resolve_project_root()
destination = PROJECT_ROOT / "cleaned_data"

if not destination.exists():
    raise FileNotFoundError("cleaned_data folder not found. Run the cleaning script first.")

print(f"Cleaned data is stored inside this project at: {destination}")


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

PROJECT_ROOT = resolve_project_root()
data_path = PROJECT_ROOT / "cleaned_data"
movies = pd.read_csv(data_path / "movies_master.csv", low_memory=False)

movies["content_features"] = movies["content_features"].fillna("")
movies["clean_title"] = movies["clean_title"].fillna("")
movies["rating_count"] = pd.to_numeric(movies["rating_count"], errors="coerce").fillna(0)
movies["rating_mean"] = pd.to_numeric(movies["rating_mean"], errors="coerce").fillna(0)

movies = movies[movies["content_features"].str.strip() != ""].copy()
movies.reset_index(drop=True, inplace=True)

print(movies.shape)


In [ ]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2)
)

tfidf_matrix = tfidf.fit_transform(movies["content_features"])
print(tfidf_matrix.shape)


In [ ]:
nn_model = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=20)
nn_model.fit(tfidf_matrix)


In [ ]:
indices = pd.Series(movies.index, index=movies["clean_title"].str.lower()).drop_duplicates()


In [ ]:
def recommend_movies(title, top_n=10):
    title = title.lower().strip()

    if title not in indices:
        return f"Movie '{title}' not found in dataset."

    idx = indices[title]

    distances, neighbors = nn_model.kneighbors(tfidf_matrix[idx], n_neighbors=top_n + 1)

    rec_indices = neighbors.flatten()[1:]
    rec_distances = distances.flatten()[1:]

    recs = movies.iloc[rec_indices][[
        "movieId",
        "clean_title",
        "year",
        "genres",
        "rating_count",
        "rating_mean",
        "director",
        "top_cast"
    ]].copy()

    recs["similarity"] = 1 - rec_distances
    recs = recs.sort_values(["similarity", "rating_count"], ascending=[False, False])

    return recs.reset_index(drop=True)


In [ ]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 1)
)


In [ ]:
recommend_movies("toy story", top_n=10)
recommend_movies("matrix, the", top_n=10)
recommend_movies("batman", top_n=10)
recommend_movies("finding nemo", top_n=10)


In [ ]:
recommend_movies("interstellar", top_n=10)


In [ ]:
import pickle
from pathlib import Path

PROJECT_ROOT = resolve_project_root()
artifacts = PROJECT_ROOT / "model_artifacts"
artifacts.mkdir(parents=True, exist_ok=True)

with (artifacts / "tfidf_vectorizer.pkl").open("wb") as file_handle:
    pickle.dump(tfidf, file_handle)

with (artifacts / "nn_model.pkl").open("wb") as file_handle:
    pickle.dump(nn_model, file_handle)

movies.to_csv(artifacts / "movies_ready.csv", index=False)

print(f"Saved successfully to: {artifacts}")


In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = resolve_project_root()
data_path = PROJECT_ROOT / "cleaned_data"
movies = pd.read_csv(data_path / "movies_master.csv", low_memory=False)






In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = resolve_project_root()
data_path = PROJECT_ROOT / "cleaned_data"
raw_data_path = PROJECT_ROOT / "raw_data" / "ml-25m"
movies = pd.read_csv(data_path / "movies_master.csv", low_memory=False)

ratings = pd.read_csv(raw_data_path / "ratings.csv", nrows=500000)

print(movies.shape)
print(ratings.shape)
ratings.head()


In [ ]:
user_counts = ratings["userId"].value_counts()
movie_counts = ratings["movieId"].value_counts()

active_users = user_counts[user_counts >= 20].index
popular_movies = movie_counts[movie_counts >= 50].index

ratings_small = ratings[
    ratings["userId"].isin(active_users) &
    ratings["movieId"].isin(popular_movies)
].copy()

print(ratings_small.shape)


In [ ]:
user_movie_matrix = ratings_small.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

print(user_movie_matrix.shape)


In [ ]:
movie_similarity = user_movie_matrix.corr(method="pearson", min_periods=20)
print(movie_similarity.shape)


In [ ]:
import pandas as pd

from pathlib import Path

PROJECT_ROOT = resolve_project_root()
movies_df = pd.read_csv(PROJECT_ROOT / "cleaned_data" / "movies_master.csv", low_memory=False)

movie_id_to_title = dict(zip(movies_df["movieId"], movies_df["clean_title"]))
title_to_movie_id = dict(zip(movies_df["clean_title"].str.lower(), movies_df["movieId"]))


In [ ]:
def recommend_collaborative(movie_title, top_n=10):
    movie_title = movie_title.lower().strip()
    
    if movie_title not in title_to_movie_id:
        return f"Movie '{movie_title}' not found."
    
    movie_id = title_to_movie_id[movie_title]
    
    if movie_id not in movie_similarity.columns:
        return f"No collaborative recommendations available for '{movie_title}'."
    
    sim_scores = movie_similarity[movie_id].dropna().sort_values(ascending=False)
    sim_scores = sim_scores.iloc[1:top_n+1]
    
    result = pd.DataFrame({
        "movieId": sim_scores.index,
        "similarity": sim_scores.values
    })
    
    result["title"] = result["movieId"].map(movie_id_to_title)
    return result[["movieId", "title", "similarity"]]


In [ ]:
recommend_collaborative("Avengers: Infinity War - Part II", top_n=10)

In [ ]:
import pandas as pd

from pathlib import Path

PROJECT_ROOT = resolve_project_root()
movies = pd.read_csv(PROJECT_ROOT / "cleaned_data" / "movies_master.csv", low_memory=False)
movies["top_cast"] = movies["top_cast"].fillna("")

def search_by_cast(actor_name):
    actor_name = actor_name.lower().strip()
    
    results = movies[movies["top_cast"].str.lower().str.contains(actor_name, na=False)].copy()
    
    return results[[
        "movieId", "clean_title", "year", "genres", "top_cast", "director", "rating_mean"
    ]].sort_values(by="rating_mean", ascending=False)

search_by_cast("tom hanks").head(20)


In [ ]:
def recommend_by_cast(actor_name, top_n=10):
    actor_name = actor_name.lower().strip()
    
    results = movies[movies["top_cast"].str.lower().str.contains(actor_name, na=False)].copy()
    
    if results.empty:
        return f"No movies found for cast member '{actor_name}'."
    
    results = results.sort_values(
        by=["rating_mean", "rating_count"],
        ascending=False
    ).head(top_n)
    
    return results[[
        "movieId", "clean_title", "year", "genres", "top_cast", "director", "rating_mean", "rating_count"
    ]]

recommend_by_cast("tom hanks", top_n=10)


In [ ]:
movies["director"] = movies["director"].fillna("")

def search_by_director(name):
    name = name.lower().strip()
    results = movies[movies["director"].str.lower().str.contains(name, na=False)].copy()
    return results[[
        "movieId", "clean_title", "year", "genres", "director", "top_cast", "rating_mean"
    ]].sort_values(by="rating_mean", ascending=False)

search_by_director("christopher nolan").head(20)


In [ ]:
import pandas as pd

from pathlib import Path

PROJECT_ROOT = resolve_project_root()
movies = pd.read_csv(PROJECT_ROOT / "cleaned_data" / "movies_master.csv", low_memory=False)

movies["clean_title"] = movies["clean_title"].fillna("")
movies["top_cast"] = movies["top_cast"].fillna("")
movies["director"] = movies["director"].fillna("")
movies["genres"] = movies["genres"].fillna("")

def search_movies(query):
    query = query.lower().strip()

    results = movies[
        movies["clean_title"].str.lower().str.contains(query, na=False) |
        movies["top_cast"].str.lower().str.contains(query, na=False) |
        movies["director"].str.lower().str.contains(query, na=False) |
        movies["genres"].str.lower().str.contains(query, na=False)
    ].copy()

    return results[[
        "movieId", "clean_title", "year", "genres",
        "top_cast", "director", "rating_mean", "rating_count"
    ]].sort_values(
        by=["rating_mean", "rating_count"],
        ascending=False
    )

search_movies("sci-fi").head(20)


In [ ]:
search_movies("christopher nolan").head(10)
